In [2]:
import duckdb
from pathlib import Path

In [3]:
import duckdb
from pipeline.endpoints import DB_PATH

# Connect to your existing persistent DuckDB database file
con = duckdb.connect(str(DB_PATH))

with con:
    # 1. Inspecting a single JSON file layout using absolute paths
    pbp_file = "../data/raw/20262027/2026020017/pbp.json"
    boxscore_file = "../data/raw/20262027/2026020017/boxscore.json"
    shifts_file = "../data/raw/20262027/2026020017/shifts.json"
    print("--- PBP SCHEMA ---")
    pbp_schema = con.execute(f"DESCRIBE SELECT * FROM read_json_auto('{pbp_file}')").fetchall()

    for column in pbp_schema:
        print(column)

    print("--- BOXSCORE SCHEMA ---")
    boxscore_schema = con.execute(f"DESCRIBE SELECT * FROM read_json_auto('{boxscore_file}')").fetchall()

    for column in boxscore_schema:
        print(column)

    print("--- SHIFTS SCHEMA ---")
    shifts_schema = con.execute(f"DESCRIBE SELECT * FROM read_json_auto('{shifts_file}')").fetchall()

    for column in shifts_schema:
        print(column)

--- PBP SCHEMA ---
('id', 'BIGINT', 'YES', None, None, None)
('season', 'BIGINT', 'YES', None, None, None)
('gameType', 'BIGINT', 'YES', None, None, None)
('limitedScoring', 'BOOLEAN', 'YES', None, None, None)
('gameDate', 'DATE', 'YES', None, None, None)
('venue', 'STRUCT("default" VARCHAR)', 'YES', None, None, None)
('venueLocation', 'STRUCT("default" VARCHAR)', 'YES', None, None, None)
('startTimeUTC', 'TIMESTAMP', 'YES', None, None, None)
('easternUTCOffset', 'VARCHAR', 'YES', None, None, None)
('venueUTCOffset', 'VARCHAR', 'YES', None, None, None)
('tvBroadcasts', 'STRUCT(id BIGINT, market VARCHAR, countryCode VARCHAR, network VARCHAR, sequenceNumber BIGINT)[]', 'YES', None, None, None)
('gameState', 'VARCHAR', 'YES', None, None, None)
('gameScheduleState', 'VARCHAR', 'YES', None, None, None)
('periodDescriptor', 'STRUCT(number BIGINT, periodType VARCHAR, maxRegulationPeriods BIGINT)', 'YES', None, None, None)
('awayTeam', 'STRUCT(id BIGINT, commonName STRUCT("default" VARCHAR), a

In [4]:
con = duckdb.connect(str(DB_PATH))
with con:
    raw_pbp = con.execute("""
                          WITH pbp_info AS (
                            SELECT 
                                id AS game_id,
                                season AS season,
                                UNNEST(plays) AS play
                            FROM read_json(?, columns:={id: 'BIGINT', season: 'BIGINT', plays: 'JSON[]'})
                          )
                          SELECT count(*)
                          FROM (
                            SELECT game_id, 
                                  season,
                                  (play->>'eventId')::BIGINT AS event_id,
                                  (play->'periodDescriptor'->>'number')::BIGINT AS period,
                                  play->'details' AS details,
                                  current_timestamp AS loaded_at,
                                  (play->>'timeInPeriod')::VARCHAR as time_in_period
                            FROM pbp_info
                          )
                         """, [str(pbp_file)]).fetchall()
    raw_pbp_limit5 = con.execute("""
                      WITH pbp_info AS (
                        SELECT 
                            id AS game_id,
                            season AS season,
                            UNNEST(plays) AS play
                        FROM read_json(?, columns:={id: 'BIGINT', season: 'BIGINT', plays: 'JSON[]'})
                      )
                        SELECT game_id, 
                              season,
                              (play->>'eventId')::BIGINT AS event_id,
                              (play->'periodDescriptor'->>'number')::BIGINT AS period,
                              (play->>'typeDescKey') AS type_desc_key,
                              (play->>'typeCode')::INTEGER AS type_code,
                              (play->>'sortOrder')::INTEGER AS sort_order,
                              (play->'periodDescriptor'->>'periodType') AS period_type,
                              (play->>'timeRemaining') AS time_remaining,
                              (play->>'situationCode') AS situation_code,
                              (play->>'homeTeamDefendingSide') AS home_team_defending_side,
                              play->'details' AS details,
                              current_timestamp AS loaded_at,
                              (play->>'timeInPeriod')::VARCHAR as time_in_period
                        FROM pbp_info
                        LIMIT 5
                      """, [str(pbp_file)]).fetchall()
    counts = con.execute("""
                    
                        SELECT len(plays)
                        FROM read_json(?, columns:={plays: 'JSON[]'})
                    
                      """, [str(pbp_file)]).fetchall()
    
    print(raw_pbp)
    print(raw_pbp_limit5)
    print(counts)

[(318,)]
[(2026020017, 20262027, 54, 1, 'period-start', 520, 9, 'REG', '20:00', '1551', 'right', None, datetime.datetime(2026, 11, 4, 10, 44, 47, 147045, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>), '00:00'), (2026020017, 20262027, 53, 1, 'faceoff', 502, 11, 'REG', '20:00', '1551', 'right', '{"eventOwnerTeamId":3,"losingPlayerId":8477429,"winningPlayerId":8476459,"xCoord":0,"yCoord":0,"zoneCode":"N"}', datetime.datetime(2026, 11, 4, 10, 44, 47, 147045, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>), '00:00'), (2026020017, 20262027, 70, 1, 'hit', 503, 12, 'REG', '19:40', '1551', 'right', '{"xCoord":7,"yCoord":41,"zoneCode":"N","eventOwnerTeamId":17,"hittingPlayerId":8477429,"hitteePlayerId":8478882}', datetime.datetime(2026, 11, 4, 10, 44, 47, 147045, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>), '00:20'), (2026020017, 20262027, 61, 1, 'shot-on-goal', 506, 19, 'REG', '19:19', '1551', 'right', '{"xCoord":92,"yCoord":19,"zoneCode":"O","

In [5]:
con = duckdb.connect(str(DB_PATH))

with con:
    boxscore_schema = con.execute(f"DESCRIBE SELECT * FROM read_json_auto('{boxscore_file}')").fetchall()

    for column in boxscore_schema:
        print(column)


('id', 'BIGINT', 'YES', None, None, None)
('season', 'BIGINT', 'YES', None, None, None)
('gameType', 'BIGINT', 'YES', None, None, None)
('limitedScoring', 'BOOLEAN', 'YES', None, None, None)
('gameDate', 'DATE', 'YES', None, None, None)
('venue', 'STRUCT("default" VARCHAR)', 'YES', None, None, None)
('venueLocation', 'STRUCT("default" VARCHAR)', 'YES', None, None, None)
('startTimeUTC', 'TIMESTAMP', 'YES', None, None, None)
('easternUTCOffset', 'VARCHAR', 'YES', None, None, None)
('venueUTCOffset', 'VARCHAR', 'YES', None, None, None)
('tvBroadcasts', 'STRUCT(id BIGINT, market VARCHAR, countryCode VARCHAR, network VARCHAR, sequenceNumber BIGINT)[]', 'YES', None, None, None)
('gameState', 'VARCHAR', 'YES', None, None, None)
('gameScheduleState', 'VARCHAR', 'YES', None, None, None)
('periodDescriptor', 'STRUCT(number BIGINT, periodType VARCHAR, maxRegulationPeriods BIGINT)', 'YES', None, None, None)
('regPeriods', 'BIGINT', 'YES', None, None, None)
('awayTeam', 'STRUCT(id BIGINT, commonNa

In [6]:
con = duckdb.connect(str(DB_PATH))
with con:
    raw_boxscore = con.execute("""
                          WITH boxscore_info AS (
                            SELECT 
                                id AS game_id,
                                playerByGameStats AS player_game_stats
                            FROM read_json(?, columns:={id: 'BIGINT', playerByGameStats: 'JSON'})
                          ),
                         players AS (
                              SELECT game_id,
                                   'away' AS team_side,
                                   'forwards' as position_group,
                                   UNNEST((player_game_stats->'awayTeam'->'forwards')::JSON[]) AS player
                              FROM boxscore_info
                              UNION ALL
                              SELECT game_id,
                                   'away' AS team_side,
                                   'defense' as position_group,
                                   UNNEST((player_game_stats->'awayTeam'->'defense')::JSON[]) AS player
                              FROM boxscore_info
                              UNION ALL
                              SELECT game_id,
                                   'away' AS team_side,
                                   'goalies' as position_group,
                                   UNNEST((player_game_stats->'awayTeam'->'goalies')::JSON[]) AS player
                              FROM boxscore_info
                              UNION ALL
                              SELECT game_id,
                                   'home' AS team_side,
                                   'forwards' as position_group,
                                   UNNEST((player_game_stats->'homeTeam'->'forwards')::JSON[]) AS player
                              FROM boxscore_info
                              UNION ALL
                              SELECT game_id,
                                   'home' AS team_side,
                                   'defense' as position_group,
                                   UNNEST((player_game_stats->'homeTeam'->'defense')::JSON[]) AS player
                              FROM boxscore_info
                              UNION ALL
                              SELECT game_id,
                                   'home' AS team_side,
                                   'goalies' as position_group,
                                   UNNEST((player_game_stats->'homeTeam'->'goalies')::JSON[]) AS player
                              FROM boxscore_info
                         )
                         SELECT
                               game_id, 
                               team_side,
                               position_group,
                               (player->>'playerId')::BIGINT AS player_id,
                               (player->'name'->>'default') AS player_name,
                               (player->>'sweaterNumber')::INTEGER as jersey_number,
                               (player->>'position') as position,
                               (player->>'toi') as toi,
                               player AS stats,
                               current_timestamp AS loaded_at
                         FROM players
                         """, [str(boxscore_file)]).fetchall()
    print(raw_boxscore)

[(2026020017, 'away', 'forwards', 8476468, 'J. Miller', 10, 'C', '17:28', '{"playerId":8476468,"sweaterNumber":10,"name":{"default":"J. Miller"},"position":"C","goals":0,"assists":0,"points":0,"plusMinus":1,"pim":0,"hits":4,"powerPlayGoals":0,"sog":2,"faceoffWinningPctg":0.333333,"toi":"17:28","blockedShots":0,"shifts":22,"giveaways":1,"takeaways":1}', datetime.datetime(2026, 11, 4, 10, 44, 47, 333893, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>)), (2026020017, 'away', 'forwards', 8482109, 'A. Lafrenière', 13, 'L', '21:30', '{"playerId":8482109,"sweaterNumber":13,"name":{"default":"A. Lafrenière"},"position":"L","goals":0,"assists":0,"points":0,"plusMinus":0,"pim":0,"hits":1,"powerPlayGoals":0,"sog":1,"faceoffWinningPctg":0.0,"toi":"21:30","blockedShots":0,"shifts":26,"giveaways":1,"takeaways":2}', datetime.datetime(2026, 11, 4, 10, 44, 47, 333893, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>)), (2026020017, 'away', 'forwards', 8481604, 'P. Dorofeyev'

In [7]:
con = duckdb.connect(str(DB_PATH))

with con:
    shifts_schema = con.execute(f"DESCRIBE SELECT * FROM read_json_auto('{shifts_file}')").fetchall()

    for column in shifts_schema:
        print(column)

('data', 'STRUCT(id BIGINT, detailCode BIGINT, duration TIME, endTime TIME, eventDescription JSON, eventDetails JSON, eventNumber BIGINT, firstName VARCHAR, gameId BIGINT, hexValue VARCHAR, lastName VARCHAR, period BIGINT, playerId BIGINT, shiftNumber BIGINT, startTime TIME, teamAbbrev VARCHAR, teamId BIGINT, teamName VARCHAR, typeCode BIGINT)[]', 'YES', None, None, None)
('total', 'BIGINT', 'YES', None, None, None)


In [8]:
con = duckdb.connect(str(DB_PATH))
with con:
    raw_shift = con.execute("""
                          WITH shift_info AS (
                            SELECT 
                                UNNEST(data) AS shifts
                            FROM read_json(?, columns:={data: 'JSON[]'})
                          )
                          SELECT count(*)
                          FROM (
                            SELECT
                              (shifts->>'gameId')::BIGINT AS game_id,
                              (shifts->>'id')::BIGINT AS shift_id,
                              (shifts->>'period')::BIGINT AS period,
                              (shifts->>'playerId')::BIGINT AS player_id,
                              (shifts->>'teamId')::INTEGER AS team_id,
                              (shifts->>'teamAbbrev') AS team_abbrev,
                              (shifts->>'shiftNumber')::INTEGER AS shift_number
                            FROM shift_info
                          )
                         """, [str(shifts_file)]).fetchall()
    raw_shift_limit5 = con.execute("""
                          WITH shift_info AS (
                            SELECT 
                                UNNEST(data) AS shifts
                            FROM read_json(?, columns:={data: 'JSON[]'})
                          )
                        SELECT
                              (shifts->>'gameId')::BIGINT AS game_id,
                              (shifts->>'id')::BIGINT AS shift_id,
                              (shifts->>'period')::BIGINT AS period,
                              (shifts->>'playerId')::BIGINT AS player_id,
                              (shifts->>'teamId')::INTEGER AS team_id,
                              (shifts->>'teamAbbrev') AS team_abbrev,
                              (shifts->>'shiftNumber')::INTEGER AS shift_number,
                              (shifts->>'startTime') AS start_time,
                              (shifts->>'endTime') AS end_time,
                              (shifts->>'duration') AS duration,
                              (shifts->>'typeCode')::BIGINT AS type_code,
                              (shifts->>'detailCode')::BIGINT AS detail_code,
                              current_timestamp AS loaded_at,
                        FROM shift_info
                        LIMIT 5
                      """, [str(shifts_file)]).fetchall()
    shifts_counts = con.execute("""
                        SELECT len(data)
                        FROM read_json(?, columns:={data: 'JSON[]'})
                      """, [str(shifts_file)]).fetchall()
    
    print(raw_shift)
    print(raw_shift_limit5)
    print(shifts_counts)

[(801,)]
[(2026020017, 16658082, 1, 8475279, 17, 'DET', 1, '01:23', '02:20', '00:57', 517, 0, datetime.datetime(2026, 11, 4, 10, 44, 47, 503892, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>)), (2026020017, 16658119, 1, 8475279, 17, 'DET', 2, '04:33', '05:12', '00:39', 517, 0, datetime.datetime(2026, 11, 4, 10, 44, 47, 503892, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>)), (2026020017, 16658120, 1, 8475279, 17, 'DET', 3, '05:55', '06:41', '00:46', 517, 0, datetime.datetime(2026, 11, 4, 10, 44, 47, 503892, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>)), (2026020017, 16658193, 1, 8475279, 17, 'DET', 4, '08:08', '08:40', '00:32', 517, 0, datetime.datetime(2026, 11, 4, 10, 44, 47, 503892, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>)), (2026020017, 16658194, 1, 8475279, 17, 'DET', 5, '10:07', '10:53', '00:46', 517, 0, datetime.datetime(2026, 11, 4, 10, 44, 47, 503892, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:0

In [9]:
# Connect to your existing persistent DuckDB database file
con = duckdb.connect(str(DB_PATH))

with con:
    # 1. Inspecting a single JSON file layout using absolute paths
    schedule_file = "../data/raw/20262027/schedule_DET.json"
    print("--- SCHELDULE SCHEMA ---")
    schedule_schema = con.execute(f"DESCRIBE SELECT * FROM read_json_auto('{schedule_file}')").fetchall()

    for column in schedule_schema:
        print(column)

--- SCHELDULE SCHEMA ---
('previousSeason', 'BIGINT', 'YES', None, None, None)
('currentSeason', 'BIGINT', 'YES', None, None, None)
('howToWatch', 'STRUCT(url VARCHAR, "text" VARCHAR)', 'YES', None, None, None)
('clubTimezone', 'VARCHAR', 'YES', None, None, None)
('clubUTCOffset', 'VARCHAR', 'YES', None, None, None)
('games', 'STRUCT(id BIGINT, season BIGINT, gameType BIGINT, gameDate DATE, venue STRUCT("default" VARCHAR, es VARCHAR, fr VARCHAR), neutralSite BOOLEAN, startTimeUTC TIMESTAMP, easternUTCOffset VARCHAR, venueUTCOffset VARCHAR, venueTimezone VARCHAR, gameState VARCHAR, gameScheduleState VARCHAR, tvBroadcasts STRUCT(id BIGINT, market VARCHAR, countryCode VARCHAR, network VARCHAR, sequenceNumber BIGINT)[], awayTeam STRUCT(id BIGINT, commonName STRUCT("default" VARCHAR, fr VARCHAR), placeName STRUCT("default" VARCHAR, fr VARCHAR), placeNameWithPreposition STRUCT("default" VARCHAR, fr VARCHAR), abbrev VARCHAR, logo VARCHAR, darkLogo VARCHAR, awaySplitSquad BOOLEAN, score BIGINT

In [10]:
team = 'DET'
con = duckdb.connect(str(DB_PATH))
with con:
    raw_schedule = con.execute("""
                          WITH schedule_info AS (
                            SELECT 
                                UNNEST(games) AS game
                            FROM read_json(?, columns:={games: 'JSON[]'})
                          )
                          SELECT count(*)
                          FROM (
                            SELECT
                              (game->>'id')::BIGINT AS game_id,
                              (game->>'season')::BIGINT AS season,
                              (game->>'gameType')::BIGINT AS game_type,
                              (game->>'gameDate') AS game_date,
                              (game->>'gameState') AS game_state,
                              (game->'awayTeam'->>'abbrev') AS away_team,
                              (game->'homeTeam'->>'abbrev') AS home_team
                            FROM schedule_info
                          )
                         """, [str(schedule_file)]).fetchall()
    raw_schedule_limit5 = con.execute("""
                          WITH schedule_info AS (
                            SELECT 
                                UNNEST(games) AS game
                            FROM read_json(?, columns:={games: 'JSON[]'})
                          )
                        SELECT
                              ? AS team,
                              (game->>'id')::BIGINT AS game_id,
                              (game->>'season')::BIGINT AS season,
                              (game->>'gameType')::BIGINT AS game_type,
                              (game->>'gameDate') AS game_date,
                              (game->>'gameState') AS game_state,
                              (game->'awayTeam'->>'abbrev') AS away_team,
                              (game->'homeTeam'->>'abbrev') AS home_team,
                              current_timestamp AS loaded_at,
                        FROM schedule_info
                        LIMIT 5
                      """, [str(schedule_file), team]).fetchall()
    raw_schedule_counts = con.execute("""
                        SELECT len(games)
                        FROM read_json(?, columns:={games: 'JSON[]'})
                      """, [str(schedule_file)]).fetchall()
    
    print(raw_schedule)
    print(raw_schedule_limit5)
    print(raw_schedule_counts)

[(88,)]
[('DET', 2026010017, 20262027, 1, '2026-09-21', 'FINAL', 'DET', 'CBJ', datetime.datetime(2026, 11, 4, 10, 44, 47, 673552, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>)), ('DET', 2026010025, 20262027, 1, '2026-09-22', 'FINAL', 'DET', 'PIT', datetime.datetime(2026, 11, 4, 10, 44, 47, 673552, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>)), ('DET', 2026010041, 20262027, 1, '2026-09-24', 'FINAL', 'BUF', 'DET', datetime.datetime(2026, 11, 4, 10, 44, 47, 673552, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>)), ('DET', 2026010061, 20262027, 1, '2026-09-26', 'FINAL', 'CBJ', 'DET', datetime.datetime(2026, 11, 4, 10, 44, 47, 673552, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>)), ('DET', 2026020017, 20262027, 2, '2026-10-02', 'OFF', 'NYR', 'DET', datetime.datetime(2026, 11, 4, 10, 44, 47, 673552, tzinfo=<DstTzInfo 'America/New_York' EST-1 day, 19:00:00 STD>))]
[(88,)]


In [12]:
con = duckdb.connect(str(DB_PATH))
with con:
    print(con.execute(
        "SELECT count(*), count(DISTINCT game_id) FROM raw.pbp_plays"
    ).fetchall())

[(3536, 11)]


In [13]:
con = duckdb.connect(str(DB_PATH))
with con:
    print(con.execute(
        "SELECT usable, count(*) FROM ingest.manifest_games WHERE season = 20262027 GROUP BY usable"
    ).fetchall())

[(0, 5), (1, 11)]


In [14]:
con = duckdb.connect(str(DB_PATH))
with con:
    print(con.execute(
        "SELECT 'pbp', count(*), count(DISTINCT game_id) FROM raw.pbp_plays UNION ALL SELECT 'box', count(*), count(DISTINCT game_id) FROM raw.boxscore_players UNION ALL SELECT 'shifts', count(*), count(DISTINCT game_id) FROM raw.shifts UNION ALL SELECT 'schedule', count(*), count(DISTINCT game_id) FROM raw.schedule"
    ).fetchall())

[('pbp', 3536, 11), ('box', 440, 11), ('shifts', 8519, 11), ('schedule', 88, 88)]


In [2]:
import duckdb
from pipeline.endpoints import DB_PATH
con = duckdb.connect(str(DB_PATH))

with con:
    # 1. Drop the table safely
    con.execute("DROP TABLE IF EXISTS raw.pbp_plays;")
    print("Table raw.pbp_plays dropped successfully.")
    
    # 2. Optional: Verify it's gone by listing remaining tables in that schema
    print("Remaining tables:", con.execute("SHOW TABLES;").fetchall())

Table raw.pbp_plays dropped successfully.
Remaining tables: []


In [2]:
import duckdb
from pipeline.endpoints import DB_PATH
con = duckdb.connect(str(DB_PATH))

with con:
    # 1. Drop the table safely
    con.execute("DROP TABLE IF EXISTS raw.schedule;")
    print("Table raw.schedule dropped successfully.")
    
    # 2. Optional: Verify it's gone by listing remaining tables in that schema
    print("Remaining tables:", con.execute("SHOW ALL TABLES;").fetchall())

Table raw.schedule dropped successfully.
Remaining tables: [('pre_scout_warehouse', 'ingest', 'manifest_games', ['game_id', 'season', 'usable', 'all_flags'], ['BIGINT', 'BIGINT', 'INTEGER', 'VARCHAR[]'], False), ('pre_scout_warehouse', 'ingest', 'manifest_latest', ['run_id', 'run_ts', 'season', 'game_id', 'endpoint', 'status', 'file_size_bytes', 'flags'], ['UUID', 'DOUBLE', 'BIGINT', 'BIGINT', 'VARCHAR', 'VARCHAR', 'BIGINT', 'VARCHAR[]'], False), ('pre_scout_warehouse', 'raw', 'boxscore_players', ['game_id', 'team_side', 'position_group', 'player_id', 'player_name', 'jersey_number', 'position', 'toi', 'stats', 'loaded_at'], ['BIGINT', 'VARCHAR', 'VARCHAR', 'BIGINT', 'VARCHAR', 'INTEGER', 'VARCHAR', 'VARCHAR', 'JSON', 'TIMESTAMP WITH TIME ZONE'], False), ('pre_scout_warehouse', 'raw', 'pbp_plays', ['game_id', 'season', 'event_id', 'period', 'type_desc_key', 'type_code', 'sort_order', 'period_type', 'time_remaining', 'situation_code', 'home_team_defending_side', 'away_team_id', 'away_tea